## **Data Engineering Task Notebook**

This notebook is designed to test you through various Data Engineering tasks using the Online Retail II dataset. The tasks will help you develop essential skills in data cleaning, feature engineering, and transforming raw data into valuable insights. You'll explore tasks like handling missing values, aggregating data, creating new features, and performing data joins to simulate real-world data workflows. These tasks will prepare you for more advanced data manipulation and analysis, enhancing your ability to work with large, complex datasets.

# **About the Dataset**
This Online Retail II dataset contains transactional data from a UK-based online retailer selling unique gift-ware. The data covers transactions that occurred between December 1, 2009 and December 9, 2011. The retailer primarily serves both individual customers and wholesalers. The products sold by the company are all-occasion gift items, including home décor, kitchenware, and other unique items.

The dataset includes detailed information on each transaction, providing valuable insights into customer behavior, sales trends, and product performance over time.



## **What can be done with this dataset?**

**Customer Behavior Analysis:** Explore purchasing patterns, repeat customers, and sales volume across different customer segments.

**Sales Forecasting:** Predict future sales by analyzing past transactions, including seasonal trends and demand fluctuations.

**Market Segmentation:** Identify customer groups based on purchase history and demographic data (e.g., by Country).

**Product Performance:** Analyze which products are bestsellers and which have low turnover, and how prices influence sales.

**Time Series Analysis:** Study trends over time, including hourly, daily, and monthly sales volumes, and identify peak shopping periods.

**Anomaly Detection:** Detect potential fraudulent transactions, cancellations, or unusually high sales activity.
Association Rule Mining: Discover products that are often purchased together and identify cross-sell opportunities.



## **Key Attributes in the Dataset:**

**InvoiceNo:** Unique transaction identifier (with cancellations indicated by 'C' prefix).

**StockCode:** Unique product code for each item sold.

**Description:** Name of the product/item sold.

**Quantity:** Quantity of each product sold in the transaction
.
**InvoiceDate:** Date and time of the transaction.

**UnitPrice:** Price per unit of the product.

**CustomerID:** Unique identifier for each customer.

**Country:** The country where the customer resides.

This dataset is a great resource for learning and practicing various data analysis, machine learning, and business intelligence techniques.

## **Exercise**
Complete the following tasks:
1. Load the [dataset](https://www.kaggle.com/datasets/lakshmi25npathi/online-retail-dataset) from Kaggle.
2. Visualize the dataset and it's structure using appropriate libraries and plots.
3. Do some basic cleaning to handle missing values
4. Create the following features:
  *   Revenue
  *   DayOfWeek: to analyze sales trends by weekdays.
  *   TotalRevenue for each CustomerID
  *   Most popular product based on Revenue.
  *   Ordersize by summing Quantity for each InvoiceNo
5. Apply a lambda function to:
  * Segment customers into tiers based on TotalRevenue (e.g., "High", "Medium", "Low").
  * Extract key information from Description and add them as columns (e.g., presence of specific keywords like "Gift" or "Discount"). At least one extra column should be added
  * Categorize transactions as "Small", "Medium", or "Large" based on Revenue.
  * **Detect Seasonal Items:** Flag items as "Christmas"-themed if the description contains relevant words.
  * Classify customers as "Loyal", "Occasional", or "One-time" based on the number of purchases.
  *  **Identify Multi-Item Invoices:** Flag invoices with multiple unique items as "Multi-Item Order".
7. Wrap all fo the above into an ETL pipeline.

Extra tasks for practicing GroupBy
1. Join CustomerID with TotalRevenue to create Customer_Revenue column
2. Group by Country to find total revenue, total customers, and average order size per country.
3. Group by StockCode to find top-selling products by quantity.
4. Group by CustomerID to calculate the average order value or frequency of purchases.







             








In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os

In [ ]:
path = kagglehub.dataset_download("lakshmi25npathi/online-retail-dataset")

print("Path to dataset files:", path)

In [ ]:
dataset_path = "C:\\Users\\nikol\\.cache\\kagglehub\\datasets\\lakshmi25npathi\\online-retail-dataset\\versions\\1"
files = os.listdir(dataset_path)
print(files)


In [ ]:
file_path = "C:\\Users\\nikol\\.cache\\kagglehub\\datasets\\lakshmi25npathi\\online-retail-dataset\\versions\\1\\online_retail_II.xlsx"
df = pd.read_excel(file_path)

In [ ]:
print(df.head())

In [ ]:
print(df.info())

In [ ]:
print(df.isnull().sum())

In [ ]:
# Fill missing 'Description' using the most common value per 'StockCode'
df['Description'] = df.groupby('StockCode')['Description'].transform(lambda x: x.fillna(x.mode()[0]) if not x.mode().empty else x)

#Drop rows where 'Customer ID' is missing
df = df.dropna(subset=['Customer ID'])

print(df.isnull().sum())


In [ ]:
print(df.duplicated().sum())

In [ ]:
df.drop_duplicates(inplace=True)
print(df.duplicated().sum())

In [ ]:
# Revenue
df["Revenue"] = df["Quantity"] * df["Price"]
print(df[["Quantity", "Price", "Revenue"]].head())


In [ ]:
# # DayOfWeek
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["weekday"] = df["InvoiceDate"].dt.day_name()
print(df["weekday"])


In [ ]:
# Total Revenue per Customer
customer_revenue = df.groupby("Customer ID")["Revenue"].sum().reset_index()
customer_revenue.columns = ["Customer ID", "Total Revenue"]
print(customer_revenue.head())

In [ ]:
# Most popular product based on revenue
product_revenue = df.groupby("Description")["Revenue"].sum().reset_index()
product_revenue = product_revenue.sort_values(by="Revenue", ascending=False)
print(product_revenue.head())


In [ ]:
# Ordersize by summing quantity for each invoiceNo
order_size = df.groupby("Description")["Quantity"].sum().reset_index()
order_size = order_size.sort_values(by="Quantity", ascending=False)
order_size.columns = ["Description", "Total Quantity"]
print(order_size.head())

In [ ]:
# 5 Apply a lamda funtions to: 
# Segment customers into tiers based on TotalRevenue (e.g., "High", "Medium", "Low").
customer_revenue["Segment"] = customer_revenue["Total Revenue"].apply(
    lambda x: "High" if x >= 1000 else "Medium" if x >= 500 else "Low"
)
print(customer_revenue.head())


In [ ]:
# Extract key information from Description and add them as columns (e.g., presence of specific keywords like "Gift" or "Discount"). At least one extra column should be added
df["Has Gift"] = df["Description"].apply(lambda x: "Gift" in str(x))
df["Has Discount"] = df["Description"].apply(lambda x: "Discount" in str(x))

print(df[["Description", "Has Gift", "Has Discount"]].head())


In [ ]:
# Categorize transactions as "Small", "Medium", or "Large" based on Revenue.
df["Transaction Size"] = df["Revenue"].apply(
    lambda x: "Large" if x >= 500 else "Medium" if x >= 100 else "Small"
)
print(df[["Revenue", "Transaction Size"]].head())

In [ ]:
# Detect Seasonal Items: Flag items as "Christmas"-themed if the description contains relevant words.
df["Is Christmas"] = df["Description"].apply(lambda x: "Christmas" in str(x))
print(df[["Description", "Is Christmas"]].head())

In [ ]:
df

In [ ]:
# Classify customers as "Loyal", "Occasional", or "One-time" based on the number of purchases.
# First, we need to calculate the number of purchases for each customer
customer_purchases = df.groupby("Customer ID")["Invoice"].nunique().reset_index()
customer_purchases.columns = ["Customer ID", "Purchase Count"]
# Now, we can classify customers based on their purchase count
customer_purchases["Customer Type"] = customer_purchases["Purchase Count"].apply(
    lambda x: "Loyal" if x >= 5 else "Occasional" if x >= 2 else "One-time"
)
print(customer_purchases.head())

In [ ]:
# Identify multi-item invoices: flag invoices with multiple unique items as "Multi-item Order".
# Check the actual column name (it might be "Invoice" instead of "InvoiceNo")
print("Available columns:", df.columns.tolist())
print("\nLooking for invoice column...")

# Use the correct column name - likely "Invoice" based on the dataset pattern
invoice_items = df.groupby("Invoice")["Description"].nunique().reset_index()
invoice_items.columns = ["Invoice", "Unique Item Count"]
invoice_items["Multi-item Order"] = invoice_items["Unique Item Count"] > 1
print(invoice_items.head())